# Classification Multinomiale des Comportements Sociaux

**Auteur :** RAMANANANDRO Andassa Fanomezantsoa (MII 227I22)  
**Institution :** EMIT Fianarantsoa  
**Approche :** Régression logistique multinomiale (softmax)

Dataset : [Social Media and Mental Health (Kaggle)](https://www.kaggle.com/datasets/souvikahmed071/social-media-and-mental-health)

Ce notebook implémente l'intégralité du pipeline : ingénierie de cible, EDA, modélisation, évaluation multicritère, simulation interactive et génération du rapport IMRAD.

## 0. Objectifs

1. Construire une cible multinomiale interprétable (4 profils).
2. Entraîner et valider un modèle softmax avec régularisation L2.
3. Interpreter les coefficients et produire le rapport Word.

## 1. Imports et configuration

In [ ]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from scipy.stats import chi2_contingency
from sklearn.metrics import classification_report, accuracy_score, f1_score

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from analysis.config import *
from analysis.data import *
from analysis.plots import *
from analysis.modeling import *
from analysis.report import generate_imrad_report

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette(PALETTE)
for d in [FIGURES_DIR, MODELS_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def save_fig(name, dpi=DPI):
    """Save current figure to outputs/figures."""
    path = FIGURES_DIR / f'{name}.png'
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches='tight')
    plt.close()
    return path


## 2. Chargement et aperçu du dataset

Le fichier `data/smmh.csv` contient 481 réponses à un questionnaire en ligne (âge, genre, usage des réseaux, échelles Likert 1–5 sur symptômes et comportements).

In [ ]:
df_raw = load_raw_data(DATA_PATH)
print(f'Dimensions : {df_raw.shape[0]} lignes x {df_raw.shape[1]} colonnes')
display(df_raw.head())
print('\n--- info ---')
df_raw.info()

In [ ]:
display(df_raw.describe(include='all').T)

### Tableau des 21 variables

| Variable | Thème |
|----------|-------|
| timestamp | Métadonnée |
| age, gender, relationship, occupation, organization | Démographie |
| uses_social_media, platforms, time_on_social_media | Usage SM |
| q9–q20 | Symptômes / comportements (Likert) |

## 3. Feature engineering — variable cible multinomiale

### 3.1 Scores composites

| Score | Items (logique) | Colonnes CSV |
|-------|-----------------|-------------|
| ADHD | Q7+Q8+Q9 | q9_no_purpose, q10_distracted, q12_easily_distracted |
| Anxiété | Q10+Q13 | q11_restless, q13_worries |
| Estime | Q15–Q17 | q15_compare, q16_comparison_feeling, q17_validation |
| Dépression | Q18–Q20 | q18_depressed, q19_interest, q20_sleep |

Références : Bergen Social Media Addiction Scale (Andreassen et al., 2012), SMUQ.

In [ ]:
df, n_ties = prepare_dataset(DATA_PATH)
print('Échantillon (utilisateurs actifs) :', len(df))
print('Cas avec égalité sur les 4 scores (résolus par rng=42) :', n_ties)
display(df[SCORE_COLS + ['behavior_profile']].describe())

### 3.2 Profil dominant (argmax) avec tie-break aléatoire reproductible

In [ ]:
display(df['behavior_profile'].value_counts().reindex(PROFILE_ORDER))
plot_class_distribution(df)

### 3.3 Test du Chi² — indépendance genre × profil

In [ ]:
ct = pd.crosstab(df['gender'], df['behavior_profile'])
chi2, p, dof, expected = chi2_contingency(ct)
print(f'Chi2={chi2:.3f}, ddl={dof}, p-value={p:.4g}')
display(ct)

## 4. Analyse exploratoire (EDA) — 8 figures publication

In [ ]:
plot_eda_figures(df)
print('Figures exportées dans outputs/figures/ (fig01–fig08, fig04 pairplot)')

## 5. Prétraitement — pipeline scikit-learn

**Anti-fuite :** les colonnes Likert q9–q20 ne sont pas dans X. Prédicteurs : âge, genre, statut, occupation, organisation, temps SM ordinal, nombre de plateformes et indicateurs binaires top plateformes.

In [ ]:
numeric_cols, categorical_cols = get_feature_columns()
feature_cols = numeric_cols + categorical_cols
X_train, X_test, y_train, y_test = split_data(df)
preprocessor = build_preprocessor(numeric_cols, categorical_cols)
fig, axes = plt.subplots(1, 2, figsize=(12,4))
y_train.value_counts().reindex(PROFILE_ORDER).plot(kind='bar', ax=axes[0], color=PALETTE, title='Train')
y_test.value_counts().reindex(PROFILE_ORDER).plot(kind='bar', ax=axes[1], color=PALETTE, title='Test')
plt.tight_layout(); plt.show()

## 6. Fondements mathématiques

### 6.1 Softmax
$$P(Y=k|\mathbf{x})=\frac{e^{\beta_k^\top\mathbf{x}}}{\sum_{j=1}^{K}e^{\beta_j^\top\mathbf{x}}}$$
Propriété : $\sum_{k=1}^{K}P(Y=k|\mathbf{x})=1$.

### 6.2 Log-vraisemblance
$$\ell(\boldsymbol{\beta})=\sum_{i=1}^{n}\sum_{k=1}^{K}\mathbb{1}[y_i=k]\log P(Y=k|\mathbf{x}_i)$$

### 6.3 Gradient et L-BFGS
$$\nabla_{\beta_k}\ell=\sum_i \mathbf{x}_i(\mathbb{1}[y_i=k]-P(Y=k|\mathbf{x}_i))$$

### 6.4 Régularisation L2
$$\ell_{reg}=\ell-\frac{\lambda}{2}\sum_k\|\beta_k\|^2,\quad C=1/\lambda$$

### 6.5–6.8 OR, métriques multiclasses, ROC OvR, LRT — appliqués en sections 9–10.

## 7. Modélisation — régression logistique multinomiale

In [ ]:
pipeline = build_pipeline(preprocessor, C=1.0)
pipeline.fit(X_train, y_train)
coef_df = extract_coefficients(pipeline)
print('Top 5 features (|coef|) par classe :')
for cls in coef_df.columns:
    print(f'\n{cls}')
    display(coef_df[cls].abs().sort_values(ascending=False).head(5))

## 8. Validation croisée et hyperparamètre C

In [ ]:
grid, cv_results, best_c = run_grid_search(build_pipeline(preprocessor), X_train, y_train)
display(cv_results)
plot_cv_results(cv_results, best_c)
print('C optimal :', best_c)

## 9. Évaluation complète

In [ ]:
pipeline_final = build_pipeline(preprocessor, C=best_c)
pipeline_final.fit(X_train, y_train)
y_pred = pipeline_final.predict(X_test)
y_proba = pipeline_final.predict_proba(X_test)
classes = list(pipeline_final.named_steps['clf'].classes_)
eval_out = plot_evaluation(y_test, y_pred, y_proba, classes)
print(classification_report(y_test, y_pred, zero_division=0))
baselines = evaluate_baselines(y_train, y_test)
print(f"Modèle — accuracy: {accuracy_score(y_test,y_pred):.3f}, F1 macro: {f1_score(y_test,y_pred,average='macro'):.3f}")
print(f"ZeroR  — accuracy: {baselines['baseline_accuracy']:.3f}, F1 macro: {baselines['baseline_f1_macro']:.3f}")

In [ ]:
proba_full = pipeline_final.predict_proba(X_train)
prior = y_train.value_counts(normalize=True).reindex(classes).to_numpy()
proba_null = np.tile(prior, (len(y_train), 1))
ll_full = log_likelihood_multinomial(y_train, proba_full, classes)
ll_null = log_likelihood_multinomial(y_train, proba_null, classes)
n_feat = pipeline_final.named_steps['prep'].fit_transform(X_train).shape[1]
lrt = compute_lrt(ll_full, ll_null, (len(classes)-1)*n_feat)
print('LRT:', lrt)

## 10. Interprétation des coefficients et odds ratios

In [ ]:
coef_df = extract_coefficients(pipeline_final)
plot_coefficients(coef_df)
or_df = compute_odds_ratios(coef_df)
plot_odds_ratios(or_df)
display(or_df.iloc[:8])

## 11. Simulation interactive

**Simulateur de Profil Comportemental — Régression Logistique Multinomiale**

Note : le modèle entraîné utilise des features exogènes ; les sliders de scores servent à une démo pédagogique.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

age_w = widgets.IntSlider(value=22, min=18, max=60, description='Âge')
time_w = widgets.IntSlider(value=4, min=1, max=9, description='Temps SM (h)')
note = widgets.HTML('<i>Prédiction à partir des features exogènes du pipeline.</i>')

def update_probs(age, time_h):
    row = X_train.iloc[[0]].copy()
    row['age'] = age
    row['time_social_media_ord'] = min(time_h, 6)
    proba = pipeline_final.predict_proba(row)[0]
    fig, ax = plt.subplots(figsize=(9,4))
    ax.bar(classes, proba, color=PALETTE)
    ax.set_ylim(0, 1)
    ax.set_ylabel('Probabilité')
    ax.set_title('Simulateur de Profil Comportemental — Régression Logistique Multinomiale')
    plt.xticks(rotation=20)
    plt.show()

ui = widgets.interactive(update_probs, age=age_w, time_h=time_w)
display(widgets.VBox([note, ui]))

## 12. Sauvegarde du modèle et rapport IMRAD

In [ ]:
from analysis.report_context import build_report_context, save_report_context
prep_t = pipeline_final.named_steps['prep'].fit_transform(X_train)
ctx = build_report_context(
    df=df, n_ties=n_ties, y_train=y_train, y_test=y_test, y_pred=y_pred,
    y_test_labels=y_test, classes=classes, pipeline_final=pipeline_final,
    coef_df=coef_df, cv_results=cv_results, best_c=best_c,
    accuracy=accuracy_score(y_test, y_pred), f1_macro=f1_score(y_test, y_pred, average='macro'),
    baselines=baselines, lrt=lrt, chi2_stat=chi2, p_chi=p, dof=dof,
    n_features_encoded=prep_t.shape[1],
)
save_report_context(ctx)
joblib.dump(pipeline_final, MODELS_DIR / 'multinomial_lr_model.pkl')
report_path = generate_imrad_report(ctx)
print('Modèle :', MODELS_DIR / 'multinomial_lr_model.pkl')
print('Rapport :', report_path)